In [1]:
import os
os.listdir("data")

['cardekho_dataset.csv']

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/cardekho_dataset.csv")
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
print(df.shape)
df.head()

(15411, 13)


,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
print(df.isna().sum())
print(df["brand"].value_counts().head(15))
df.describe()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64
brand
Maruti           4992
Hyundai          2982
Honda            1485
Mahindra         1011
Toyota            793
Ford              790
Volkswagen        620
Renault           536
BMW               439
Tata              430
Mercedes-Benz     337
Skoda             334
Audi              192
Datsun            170
Jaguar             59
Name: count, dtype: int64


,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
count,15411.000000,1.541100e+04,15411.000000,15411.000000,15411.000000,15411.000000,1.541100e+04
mean,6.036338,5.561648e+04,19.701151,1486.057751,100.588254,5.325482,7.749711e+05
std,3.013291,5.161855e+04,4.171265,521.106696,42.972979,0.807628,8.941284e+05
min,0.000000,1.000000e+02,4.000000,793.000000,38.400000,0.000000,4.000000e+04
25%,4.000000,3.000000e+04,17.000000,1197.000000,74.000000,5.000000,3.850000e+05
50%,6.000000,5.000000e+04,19.670000,1248.000000,88.500000,5.000000,5.560000e+05
75%,8.000000,7.000000e+04,22.700000,1582.000000,117.300000,5.000000,8.250000e+05
max,29.000000,3.800000e+06,33.540000,6592.000000,626.000000,9.000000,3.950000e+07


In [4]:
before = len(df)
df = df[(df.km_driven > 100) & (df.km_driven < 500_000)]
df = df[
    df.selling_price.between(
        df.selling_price.quantile(0.005), df.selling_price.quantile(0.995)
    )
]
print(f"{before} -> {len(df)} rows")

15411 -> 15245 rows


In [5]:
specs = (
    df.groupby(["brand", "model"])[["engine", "max_power", "mileage", "seats"]]
    .median()
    .reset_index()
)
specs.to_json("specs.json", orient="records")
print(len(specs), "brand-model combos")

115 brand-model combos


In [6]:
from sklearn.model_selection import train_test_split

features = [
    "brand",
    "model",
    "vehicle_age",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "engine",
    "max_power",
    "mileage",
    "seats",
]
cat = ["brand", "model", "fuel_type", "transmission_type"]

X = df[features].copy()
for c in cat:
    X[c] = X[c].astype("category")
y = np.log(df.selling_price)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_tr.shape, X_te.shape)

(12196, 10) (3049, 10)


In [7]:
import lightgbm as lgb

models = {}
for q in [0.1, 0.5, 0.9]:
    m = lgb.LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        verbose=-1,
    )
    m.fit(X_tr, y_tr)
    models[q] = m
print("trained")

trained


In [8]:
p10, p50, p90 = (np.exp(models[q].predict(X_te)) for q in [0.1, 0.5, 0.9])
actual = np.exp(y_te).values

mae = np.mean(np.abs(p50 - actual))
mape = np.mean(np.abs(p50 - actual) / actual) * 100
coverage = np.mean((actual >= p10) & (actual <= p90)) * 100
print(
    f"LightGBM  -> MAE ₹{mae:,.0f} | MAPE {mape:.1f}% | P10–P90 coverage {coverage:.1f}%"
)

LightGBM  -> MAE ₹82,168 | MAPE 12.3% | P10–P90 coverage 74.2%


In [9]:
train_df = df.loc[X_tr.index]
med_by_model = train_df.groupby("model")["selling_price"].median()
global_med = train_df["selling_price"].median()

base = df.loc[X_te.index, "model"].map(med_by_model).fillna(global_med).values
b_mae = np.mean(np.abs(base - actual))
b_mape = np.mean(np.abs(base - actual) / actual) * 100
print(f"Baseline  -> MAE ₹{b_mae:,.0f} | MAPE {b_mape:.1f}%")
print(f"Improvement: {100*(b_mae-mae)/b_mae:.0f}% lower MAE")

Baseline  -> MAE ₹191,941 | MAPE 28.8%
Improvement: 57% lower MAE


In [10]:
import joblib

joblib.dump(
    {
        "models": models,
        "features": features,
        "categories": {c: list(X[c].cat.categories) for c in cat},
    },
    "price_model.joblib",
)
print("saved")

saved


In [11]:
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=42
)

cqr_models = {}
for q in [0.1, 0.5, 0.9]:
    m = lgb.LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        verbose=-1,
    )
    m.fit(X_fit, y_fit)
    cqr_models[q] = m

lo, hi = cqr_models[0.1].predict(X_cal), cqr_models[0.9].predict(X_cal)
scores = np.maximum(lo - y_cal.values, y_cal.values - hi)
n = len(scores)
qhat = np.quantile(scores, min(1.0, 0.8 * (1 + 1 / n)))
print(f"calibration offset (log scale): {qhat:.4f}")

calibration offset (log scale): 0.0252


In [12]:
p10c = np.exp(cqr_models[0.1].predict(X_te) - qhat)
p50c = np.exp(cqr_models[0.5].predict(X_te))
p90c = np.exp(cqr_models[0.9].predict(X_te) + qhat)

mape_c = np.mean(np.abs(p50c - actual) / actual) * 100
cov_c = np.mean((actual >= p10c) & (actual <= p90c)) * 100
width = np.median((p90c - p10c) / p50c) * 100
print(
    f"CQR -> MAPE {mape_c:.1f}% | coverage {cov_c:.1f}% | median band width ±{width/2:.0f}%"
)

CQR -> MAPE 12.4% | coverage 80.3% | median band width ±18%


In [13]:
joblib.dump(
    {
        "models": cqr_models,
        "qhat": float(qhat),
        "features": features,
        "categories": {c: list(X[c].cat.categories) for c in cat},
    },
    "price_model.joblib",
)
print("saved calibrated model")

saved calibrated model
